# LMIP Schema Initialization

**Purpose**: Create all required schemas for the Labour Market Intelligence Platform

**Target Catalog**: `workspace`

## Schema Architecture

This notebook creates the complete schema structure following the LMIP multi-schema medallion architecture:

| Schema | Purpose |
|--------|----------|
| **metadata** | Source configurations, DQ rules, taxonomy, run control |
| **bronze** | Raw ingestion layer (API snapshots, response logs) |
| **silver** | Cleansed and deduplicated jobs data |
| **semantic** | Enriched semantic layer (role mapping, skills, company canonical) |
| **warehouse** | Star schema dimensional model (dims + facts) |
| **gold** | Pre-aggregated business metrics and trends |
| **audit** | Pipeline runs, DQ results, access events |
| **publish** | Consumer-ready datasets and reports |
| **quarantine** | Failed records and data quality violations |

---

## Usage

**First-time initialization**:
```python
dbutils.notebook.run("/LMIP/notebooks/init/init_create_schemas", timeout_seconds=300)
```

**Idempotent**: Safe to run multiple times (uses IF NOT EXISTS)

In [0]:
# Databricks notebook source
# Target catalog for all LMIP schemas
CATALOG = "workspace"

# Schema definitions with descriptions
SCHEMAS = [
    ("metadata", "Source configurations, DQ rules, taxonomy mappings, and pipeline run control"),
    ("bronze", "Raw ingestion layer - API snapshots and response logs"),
    ("silver", "Cleansed and deduplicated job postings with change tracking"),
    ("semantic", "Semantic enrichment - role mapping, skill catalog, company canonicalization"),
    ("warehouse", "Star schema dimensional model with conformed dimensions and facts"),
    ("gold", "Pre-aggregated business metrics, KPIs, and trend analyses"),
    ("audit", "Pipeline runs, DQ results, access logs, and compliance tracking"),
    ("publish", "Consumer-ready datasets, reports, and external integrations"),
    ("quarantine", "Failed records, DQ violations, and rejected data for investigation")
]

print(f"Target Catalog: {CATALOG}")
print(f"Schemas to create: {len(SCHEMAS)}")

In [0]:
%sql
-- Create all LMIP platform schemas
-- Idempotent: safe to run multiple times

CREATE SCHEMA IF NOT EXISTS metadata 
  COMMENT 'Source configurations, DQ rules, taxonomy mappings, and pipeline run control';

CREATE SCHEMA IF NOT EXISTS bronze 
  COMMENT 'Raw ingestion layer - API snapshots and response logs';

CREATE SCHEMA IF NOT EXISTS silver 
  COMMENT 'Cleansed and deduplicated job postings with change tracking';

CREATE SCHEMA IF NOT EXISTS semantic 
  COMMENT 'Semantic enrichment - role mapping, skill catalog, company canonicalization';

CREATE SCHEMA IF NOT EXISTS warehouse 
  COMMENT 'Star schema dimensional model with conformed dimensions and facts';

CREATE SCHEMA IF NOT EXISTS gold 
  COMMENT 'Pre-aggregated business metrics, KPIs, and trend analyses';

CREATE SCHEMA IF NOT EXISTS audit 
  COMMENT 'Pipeline runs, DQ results, access logs, and compliance tracking';

CREATE SCHEMA IF NOT EXISTS publish 
  COMMENT 'Consumer-ready datasets, reports, and external integrations';

CREATE SCHEMA IF NOT EXISTS quarantine 
  COMMENT 'Failed records, DQ violations, and rejected data for investigation';

SELECT 'All schemas created successfully' AS status;

In [0]:
# Databricks notebook source
from datetime import datetime, timezone

# Query to verify all schemas exist
verify_query = f"""
SELECT databaseName as schema_name
FROM {CATALOG}.information_schema.schemata
WHERE databaseName IN ('metadata', 'bronze', 'silver', 'semantic', 'warehouse', 'gold', 'audit', 'publish', 'quarantine')
ORDER BY databaseName
"""

schemas_df = spark.sql(verify_query)
found_schemas = [row.schema_name for row in schemas_df.collect()]

print("\n" + "="*60)
print("SCHEMA VERIFICATION REPORT")
print("="*60)
print(f"Timestamp: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"Catalog: {CATALOG}")
print()

expected_schemas = [schema[0] for schema in SCHEMAS]
missing_schemas = set(expected_schemas) - set(found_schemas)

if not missing_schemas:
    print("✓ All schemas created successfully")
    print(f"✓ Total schemas: {len(found_schemas)}")
    print()
    print("Created schemas:")
    for schema in sorted(found_schemas):
        print(f"  • {CATALOG}.{schema}")
    status = "SUCCESS"
else:
    print("✗ Schema creation incomplete")
    print(f"✗ Missing schemas: {', '.join(sorted(missing_schemas))}")
    status = "FAILED"

print("="*60)

# Return status for orchestration
dbutils.notebook.exit(status)